In [1]:
!pip install torch

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Load data
text = """hello world. this is a tiny gpt built from scratch.
it learns how to predict the next character."""

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [22]:
data

tensor([10,  7, 12, 12, 15,  1, 21, 15, 17, 12,  6,  2,  1, 19, 10, 11, 18,  1,
        11, 18,  1,  3,  1, 19, 11, 14, 23,  1,  9, 16, 19,  1,  4, 20, 11, 12,
        19,  1,  8, 17, 15, 13,  1, 18,  5, 17,  3, 19,  5, 10,  2,  0, 11, 19,
         1, 12,  7,  3, 17, 14, 18,  1, 10, 15, 21,  1, 19, 15,  1, 16, 17,  7,
         6, 11,  5, 19,  1, 19, 10,  7,  1, 14,  7, 22, 19,  1,  5, 10,  3, 17,
         3,  5, 19,  7, 17,  2])

In [3]:
# Hyperparameters

batch_size = 16
block_size = 32
n_embd = 64
n_head = 4
n_layer = 2
dropout = 0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
# Data Loader

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [5]:
# Step 5: Self-Attention (core of GPT)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v


In [6]:
# Step 6: Multi-Head Attention + Feed Forward

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


In [7]:
# Step 7: Transformer Block

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [8]:
# Step 8: GPT Model

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx


In [9]:
# Step 9: Train

model = GPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(1000):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"step {step} | loss {loss.item():.4f}")


step 0 | loss 3.3212
step 100 | loss 0.7899
step 200 | loss 0.1816
step 300 | loss 0.0972
step 400 | loss 0.0613
step 500 | loss 0.0610
step 600 | loss 0.0766
step 700 | loss 0.0706
step 800 | loss 0.0512
step 900 | loss 0.0630


In [10]:
# Step 10: Generate text 🎉

context = torch.zeros((1,1), dtype=torch.long).to(device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))



it learns how to predict the next character.
it lerns how to predict the next character.
ict tho ch.
itit learns how to predict the next character.
it hch.
it learns how to predict the next character.
